# TP : prédire le cours du CAC 40 avec des réseaux récurrents

## 1.0 Les cours

Cours de clôture quotidiens du CAC 40, en points, de janvier 2005 à novembre 2026. On entraîne sur les jours jusqu'à
fin 2017 et on valide sur les suivants : pas de tirage aléatoire, un modèle ne doit jamais s'entraîner sur un jour
postérieur à ceux sur lesquels on le juge.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
keras.utils.set_random_seed(0)

close = pd.read_csv("cac40.csv", index_col="date", parse_dates=True)["close"]
CUT = "2018-01-01"
train_close, val_close = close[close.index < CUT], close[close.index >= CUT]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
train_close.plot(ax=ax, label="entraînement")
val_close.plot(ax=ax, label="validation")
ax.axhline(train_close.max(), color="gray", linestyle=":", label="maximum de l'entraînement")
ax.set_ylabel("CAC 40 (points)")
ax.legend()
plt.tight_layout()
plt.show()

**Q1.** Un prévisionniste paresseux répond chaque jour : « demain, le CAC 40 clôturera où il a clôturé
aujourd'hui ». Quelle erreur typique attendez-vous de lui, comparée au niveau de l'indice ? Pourquoi sera-t-il
difficile à battre ?

**Votre réponse :**




## 1.1 Des fenêtres de cours

Le modèle lit les $W$ derniers cours de clôture et prédit le suivant. Une fenêtre a la forme $(W, 1)$ : $W$ jours, une
valeur par jour. Les cours sont ensuite ramenés entre 0 et 1 avec le minimum et le maximum de l'entraînement.

In [ ]:
def make_windows(series, window):
    """All pairs (`window` consecutive values, the value right after them) of one series."""
    values = series.to_numpy()
    # TODO: for every start i that leaves `window` values and one more inside `values`:
    # x = values[i : i + window] and y = values[i + window];
    # return x with shape (n, window, 1) and y with shape (n,), both float32
    ...


x, y = make_windows(close, 20)
assert x.shape == (len(close) - 20, 20, 1) and y.shape == (len(close) - 20,)
assert np.allclose(x[0, :, 0], close.iloc[:20]) and np.isclose(y[0], close.iloc[20])

In [ ]:
WINDOW = 20
X_train, y_train = make_windows(train_close, WINDOW)
X_val, y_val = make_windows(val_close, WINDOW)
print(f"entraînement : {X_train.shape}, validation : {X_val.shape}")

In [ ]:
lo, hi = train_close.min(), train_close.max()


def scale(values):
    """Min-max scaling of prices with the training minimum and maximum."""
    # TODO: 0 at `lo`, 1 at `hi`, float32
    ...


assert np.allclose(scale(np.array([lo, hi])), [0, 1])
Xs_train, ys_train, Xs_val, ys_val = scale(X_train), scale(y_train), scale(X_val), scale(y_val)

## 1.2 La référence « demain = aujourd'hui »

C'est la prévision à battre : elle ne sait rien de l'avenir et n'apprend rien.

In [ ]:
# TODO: mean absolute error, in index points, of "tomorrow's close = today's close" on each set of windows:
# the last value of each window against its target
mae_persistence_train = ...
mae_persistence_val = ...
assert np.isclose(mae_persistence_val, mean_absolute_error(y_val, X_val[:, -1, 0]))
print(f"« demain = aujourd'hui » : {mae_persistence_train:.0f} points en entraînement, {mae_persistence_val:.0f} en validation")

## 1.3 Un RNN simple

Une couche `SimpleRNN(32)` lit la fenêtre jour après jour. À chaque jour elle met à jour son état
$h_t = \tanh(x_t W_x + h_{t-1} W_h + b)$, avec $h_0 = 0$, et l'état $h_W$ du dernier jour passe dans une
couche `Dense(1)` qui sort le cours prévu.

**Q2.** Combien de paramètres compte la couche `SimpleRNN(32)` pour une valeur par jour ? Ce nombre change-t-il avec la
longueur $W$ de la fenêtre ?

**Votre réponse :**




In [ ]:
def build_model(cell, units, window):
    """A recurrent layer reading `window` days (one value per day), then one output: the next value."""
    # TODO: a keras.Sequential made of layers.Input(shape=(window, 1)), cell(units) and a layers.Dense(1) layer
    ...


rnn = build_model(layers.SimpleRNN, 32, WINDOW)
# TODO: the number of parameters of the SimpleRNN layer, from your answer to Q2
rnn_params = ...
assert rnn_params == rnn.layers[0].count_params()
print(f"couche SimpleRNN : {rnn_params} paramètres, modèle entier : {rnn.count_params()}")

## 1.4 La récurrence à la main

La couche calcule la formule ci-dessus, rien de plus. Refaites-la sur une fenêtre avec les poids de la couche.

In [ ]:
W_x, W_h, b = rnn.layers[0].get_weights()  # shapes (1, 32), (32, 32) and (32,)

h = np.zeros(32)
for x_t in Xs_val[0]:  # the 20 days of one window, one at a time
    # TODO: the new state h from x_t, W_x, W_h, b and the previous h (the kernels are applied as x_t @ W_x and h @ W_h)
    h = ...
assert np.allclose(h, keras.ops.convert_to_numpy(rnn.layers[0](Xs_val[:1]))[0], atol=1e-5)

## 1.5 Prévoir le cours avec le RNN simple

On entraîne le RNN sur les cours mis à l'échelle, puis on ramène ses prévisions en points.

In [ ]:
def train(model, X_train, y_train, X_val, y_val, epochs):
    """Trains on the training windows, validating after each epoch. Returns the cost and error of each epoch."""
    # TODO: compile with the "adam" optimizer, the "mse" loss and the "mae" metric; then fit for `epochs` epochs,
    # batches of 32, validation_data=(X_val, y_val), verbose=0; return the .history of what fit returns
    ...


def predict_points(model, Xs):
    """Forecast of the model for scaled windows, in index points."""
    # TODO: model.predict (verbose=0) gives scaled values of shape (n, 1): flatten them and undo `scale`
    ...


def plot_curves(histories, reference=None):
    """Cost (MSE) and mean absolute error after each epoch, training (solid) and validation (dashed), one color per model."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for (name, history), color in zip(histories.items(), plt.cm.tab10.colors):
        for ax, metric in zip(axes, ["loss", "mae"]):
            ax.plot(history[metric], color=color, label=f"{name} (entraînement)")
            ax.plot(history[f"val_{metric}"], color=color, linestyle="--", label=f"{name} (validation)")
    for (name, mse), color in zip((reference or {}).items(), ["gray", "black"]):
        axes[0].axhline(mse, color=color, linestyle=":", label=name)
    for ax, label in zip(axes, ["erreur quadratique moyenne", "erreur absolue moyenne"]):
        ax.set_xlabel("époque")
        ax.set_ylabel(label)
    axes[0].legend()
    plt.tight_layout()
    plt.show()


def plot_forecasts(preds):
    """Closes of the validation period and the forecast of each model."""
    days = val_close.index[WINDOW:]
    plt.figure(figsize=(11, 3.8))
    plt.plot(days, y_val, color="black", label="cours réel")
    for (name, pred), color in zip(preds.items(), plt.cm.tab10.colors):
        plt.plot(days, pred, color=color, label=name)
    plt.axhline(train_close.max(), color="gray", linestyle=":", label="maximum de l'entraînement")
    plt.ylabel("CAC 40 (points)")
    plt.legend()
    plt.tight_layout()
    plt.show()


history_rnn = train(rnn, Xs_train, ys_train, Xs_val, ys_val, epochs=30)
assert len(history_rnn["val_loss"]) == 30
plot_curves({"RNN": history_rnn})
pred_rnn = predict_points(rnn, Xs_val)
assert pred_rnn.shape == y_val.shape
print(f"RNN : {mean_absolute_error(y_train, predict_points(rnn, Xs_train)):.0f} points en entraînement, "
      f"{mean_absolute_error(y_val, pred_rnn):.0f} en validation")
plot_forecasts({"RNN": pred_rnn})

**Q3.** Comparez l'erreur du RNN en entraînement et en validation à celle de « demain = aujourd'hui », et
regardez sur la figure où la prévision s'écarte du cours. Que s'est-il passé ?

**Votre réponse :**




## 1.6 LSTM et GRU

Même modèle, avec `layers.LSTM` puis `layers.GRU` à la place de `layers.SimpleRNN`, 32 unités chacun. Le LSTM
calcule quatre quantités par jour (portes d'oubli, d'entrée et de sortie, et candidat), le GRU trois (portes de mise
à jour et de réinitialisation, et candidat), le RNN simple une seule.

**Q4.** Sans exécuter : rangez les trois couches par nombre de paramètres, avec leurs rapports à celui du RNN
simple.

**Votre réponse :**




In [ ]:
cells = {"RNN": layers.SimpleRNN, "LSTM": layers.LSTM, "GRU": layers.GRU}
n_params, preds = {}, {}
for name, cell in cells.items():
    keras.utils.set_random_seed(0)
    # TODO: build the model with 32 units of this cell on WINDOW days and train it for 30 epochs on the scaled windows;
    # store the number of parameters of its recurrent layer in n_params[name] and its forecasts of the
    # validation closes, in index points, in preds[name]
    ...

assert set(preds) == set(cells) and n_params["RNN"] == rnn_params
plot_forecasts(preds)
for name in cells:
    print(f"{name:4s} {n_params[name]:5d} paramètres, {mean_absolute_error(y_val, preds[name]):.0f} points en validation "
          f"(« demain = aujourd'hui » : {mae_persistence_val:.0f})")

**Q5.** Les portes corrigent-elles ce que le RNN simple faisait mal ? Que conclure sur le choix de la cellule ?

**Votre réponse :**




## 1.7 Prévoir la variation plutôt que le cours

Les cours futurs sortent de la plage vue à l'entraînement (figure de 1.0) ; la variation d'un jour, le **rendement**
$r_t = 100 \ln (P_t / P_{t-1})$ en pourcentage, reste dans la même plage d'une année à l'autre. Mêmes modèles, mêmes
fenêtres de 20 jours, mais sur les rendements : on prédit le rendement de demain, puis on en tire un cours.

**Q6.** Si les 20 derniers rendements permettaient de prévoir celui de demain, que ferait le marché ?

**Votre réponse :**




In [ ]:
# TODO: daily log returns in percent of `close` (np.log, Series.diff), without the empty first row
returns = ...
r_train, r_val = returns[returns.index < CUT], returns[returns.index >= CUT]
Xr_train, yr_train = make_windows(r_train, WINDOW)
Xr_val, yr_val = make_windows(r_val, WINDOW)
assert Xr_val.shape[0] == X_val.shape[0] and Xr_train.shape[0] == X_train.shape[0] - 1

# TODO: `mse_zero`: mean squared error on yr_val of "tomorrow's return is 0";
# `rate_up`: share of days with a positive return in yr_val, the directional accuracy of "tomorrow goes up"
mse_zero = ...
rate_up = ...
assert np.isclose(mse_zero, mean_squared_error(yr_val, np.zeros_like(yr_val))) and 0.4 < rate_up < 0.6
print(f"rendement nul : erreur quadratique {mse_zero:.3f} ; « demain, hausse » : juste {rate_up:.1%} des jours")

In [ ]:
histories_r, preds_r = {}, {}
for name, cell in cells.items():
    keras.utils.set_random_seed(0)
    # TODO: build the model with 32 units of this cell on WINDOW days and train it for 20 epochs on (Xr_train, yr_train),
    # validated on (Xr_val, yr_val); store its history in histories_r[name] and its forecasts of the
    # validation returns, a vector of shape (n,), in preds_r[name]
    ...

# TODO: from each return forecast, the forecast closing price in index points: the last price of the window
# (X_val[:, -1, 0]) times exp(forecast / 100)
price_forecasts = {name: ... for name, pred in preds_r.items()}

plot_curves(histories_r, {"rendement nul": mse_zero})
for name, pred in preds_r.items():
    print(f"{name:4s} erreur quadratique {mean_squared_error(yr_val, pred):.3f}, sens juste {np.mean((pred > 0) == (yr_val > 0)):.1%}, "
          f"cours : {mean_absolute_error(y_val, price_forecasts[name]):.1f} points (« demain = aujourd'hui » : {mae_persistence_val:.1f})")

**Q7.** Lisez les courbes et le tableau. Un des modèles bat-il « rendement nul » ? Fait-il mieux que « demain,
hausse » sur le sens ? Que vaut son cours prévu face à « demain = aujourd'hui » ?

**Votre réponse :**




## 1.8 Ce que coûte l'erreur

Une règle de trading : on est acheteur le lendemain si le modèle prévoit une hausse, vendeur à découvert sinon. Le
gain du jour est la position (+1 ou −1) fois le rendement. Chaque unité de position achetée ou vendue coûte 0,05 % du montant : passer d'acheteur à vendeur en vaut deux.

**Q8.** Un modèle juste sur le sens 52 jours sur 100 est-il rentable avec cette règle ? Que faut-il calculer pour le
savoir ?

**Votre réponse :**




In [ ]:
def strategy_returns(pred, y, cost=0.05):
    """Daily returns (%) of the rule "long if the forecast is positive, short otherwise", paying `cost` % per unit of position change."""
    # TODO: position = +1 if pred > 0 else -1 (np.where); gain = position * y; the cost is paid on the absolute change
    # of position from the day before (the first day pays |position|); return gain minus cost
    ...


assert np.isclose(strategy_returns(np.ones(len(yr_val)), yr_val, cost=0).sum(), yr_val.sum())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, cost, title in zip(axes, [0, 0.05], ["sans frais", "frais de 0,05 % par unité échangée"]):
    ax.plot(r_val.index[WINDOW:], np.cumsum(yr_val), color="black", label="acheter et garder")
    for (name, pred), color in zip(preds_r.items(), plt.cm.tab10.colors):
        ax.plot(r_val.index[WINDOW:], np.cumsum(strategy_returns(pred, yr_val, cost)), color=color, label=name)
    ax.set_title(title)
    ax.set_ylabel("rendement cumulé (%)")
axes[0].legend()
plt.tight_layout()
plt.show()

**Q9.** Lisez la figure. Utiliseriez-vous un de ces modèles pour acheter et vendre le CAC 40 ? Que conclure sur
la prévision du cours par un réseau récurrent ?

**Votre réponse :**


